# AdventureWorks Sales – Data Cleaning Pipeline

> Reusable pandas-based cleaning function for AdventureWorks sales query outputs (type conversion, missing values, status filtering, and edge-case handling). 
This notebook defines and tests a reusable `clean_sales_dataframe` function for AdventureWorks sales query outputs.
It focuses on:
- Date and numeric type coercion
- Handling missing values
- Optional filtering of cancelled/invalid orders

In [2]:
# Setup & imports

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## Helper function: `clean_sales_dataframe`

This section defines a reusable cleaning function for AdventureWorks sales query outputs (dates, numeric columns, and optional status-based filtering).

In [3]:
def clean_sales_dataframe(
    df,
    date_col="OrderDate",
    numeric_cols=None,
    status_col=None,
    exclude_status=None,
    start_date=None,
    end_date=None
):
    """
    Clean and standardize an AdventureWorks sales query result.

    Steps:
    - Convert `date_col` to datetime, dropping rows where conversion fails.
    - Optionally filter to a date range [`start_date`, `end_date`] (inclusive).
    - Coerce specified numeric columns (e.g., SubTotal, LineTotal, OrderQty)
      to numeric, setting non-parsable values to NaN.
    - Optionally drop rows where all specified numeric columns are missing
      after coercion.
    - Optionally filter out rows whose `status_col` is in `exclude_status`
      (e.g., Status values 6 = Cancelled, or other custom invalid codes from
      the source query). [web:6][web:15]

    Edge cases / assumptions:
    - If `date_col` is missing or cannot be converted for a row, that row is removed.
    - If a numeric column in `numeric_cols` is missing from `df`, it is silently ignored.
    - If `status_col` is provided but not present in `df`, no status-based filtering is applied.
    - `exclude_status` can be a single value or an iterable of values; if None,
      no status filtering is done.
    - Status semantics follow AdventureWorks SalesOrderHeader:
      1 = In process; 2 = Approved; 3 = Backordered; 4 = Rejected;
      5 = Shipped; 6 = Cancelled. [web:6][web:52]
    """
    import pandas as pd

    df = df.copy()

    # 1) Date conversion
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        df = df.dropna(subset=[date_col])
    else:
        raise KeyError(f"{date_col} not found in DataFrame columns.")

    # 2) Date range filter
    if start_date is not None:
        df = df[df[date_col] >= pd.to_datetime(start_date)]
    if end_date is not None:
        df = df[df[date_col] <= pd.to_datetime(end_date)]

    # 3) Numeric coercion
    if numeric_cols is None:
        numeric_cols = ["SubTotal", "LineTotal", "OrderQty"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")  # invalid -> NaN [web:12][web:56]

    # Drop rows where all numeric columns are NaN (if any numeric columns exist)
    present_numeric = [c for c in numeric_cols if c in df.columns]
    if present_numeric:
        df = df.dropna(subset=present_numeric, how="all")

    # 4) Status filter (cancelled/invalid)
    if status_col is not None and status_col in df.columns and exclude_status is not None:
        if not isinstance(exclude_status, (list, set, tuple)):
            exclude_values = {exclude_status}
        else:
        # Normalize exclude_status to a set
            exclude_values = set(exclude_status)

        df = df[~df[status_col].isin(exclude_values)]

    # 5) Final tidy-up
    df = df.reset_index(drop=True)
    return df



In [4]:
import sqlalchemy as sa
import urllib
import pandas as pd

# Connection details
server = r"MYDELL23\SQLEXPRESS01"
database = "AdventureWorks"

conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    f"Server={server};"
    f"Database={database};"
    "Trusted_Connection=yes;"
)

params = urllib.parse.quote_plus(conn_str)
engine = sa.create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query against AdventureWorks and return a DataFrame."""
    return pd.read_sql(sql, engine)

In [5]:
sql_sales_lines = """
SELECT
    soh.OrderDate,
    p.Name AS ProductName,
    sod.LineTotal,
    sod.OrderQty,
    soh.Status
FROM Sales.SalesOrderDetail AS sod
JOIN Sales.SalesOrderHeader AS soh
    ON sod.SalesOrderID = soh.SalesOrderID
JOIN Production.Product AS p
    ON sod.ProductID = p.ProductID;
"""

df_sales_lines = run_query(sql_sales_lines)

clean_df_sales_lines = clean_sales_dataframe(
    df_sales_lines,
    date_col="OrderDate",
    numeric_cols=["LineTotal", "OrderQty"],
    status_col="Status",
    exclude_status=[6]
)

clean_df_sales_lines.info()
clean_df_sales_lines.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121317 entries, 0 to 121316
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   OrderDate    121317 non-null  datetime64[ns]
 1   ProductName  121317 non-null  object        
 2   LineTotal    121317 non-null  float64       
 3   OrderQty     121317 non-null  int64         
 4   Status       121317 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 4.6+ MB


,OrderDate,ProductName,LineTotal,OrderQty,Status
0,2011-05-31,"Mountain-100 Black, 42",2024.994,1,5
1,2011-05-31,"Mountain-100 Black, 44",6074.982,3,5
2,2011-05-31,"Mountain-100 Black, 48",2024.994,1,5
3,2011-05-31,"Mountain-100 Silver, 38",2039.994,1,5
4,2011-05-31,"Mountain-100 Silver, 42",2039.994,1,5


In [18]:
clean_df_sales_lines = clean_sales_dataframe(
    df_sales_lines,
    date_col="OrderDate",
    numeric_cols=["LineTotal", "OrderQty"],
    status_col="Status",
    exclude_status=[6]   # drop cancelled
)

clean_df_sales_lines.columns.tolist()
clean_df_sales_lines.head()


,OrderDate,ProductName,LineTotal,OrderQty,Status
0,2011-05-31,"Mountain-100 Black, 42",2024.994,1,5
1,2011-05-31,"Mountain-100 Black, 44",6074.982,3,5
2,2011-05-31,"Mountain-100 Black, 48",2024.994,1,5
3,2011-05-31,"Mountain-100 Silver, 38",2039.994,1,5
4,2011-05-31,"Mountain-100 Silver, 42",2039.994,1,5


In [19]:
clean_df_sales_lines.columns.tolist()


['OrderDate', 'ProductName', 'LineTotal', 'OrderQty', 'Status']

In [20]:
df = clean_df_sales_lines.copy()

df["ProductCategoryName"] = df["ProductCategoryName"].str.strip()

df["is_bike"] = df["ProductCategoryName"].eq("Bikes")
df["is_accessory"] = df["ProductCategoryName"].eq("Accessories")
df["is_clothing"] = df["ProductCategoryName"].eq("Clothing")
df["is_components"] = df["ProductCategoryName"].eq("Components")

KeyError: 'ProductCategoryName'

In [7]:
clean_df_sales_lines.describe(include="all")

,OrderDate,ProductName,LineTotal,OrderQty,Status
count,121317,121317,121317.000000,121317.000000,121317.0
unique,NaN,266,NaN,NaN,NaN
top,NaN,Water Bottle - 30 oz.,NaN,NaN,NaN
freq,NaN,4688,NaN,NaN,NaN
mean,2013-07-15 19:44:52.321768704,NaN,905.449207,2.266080,5.0
min,2011-05-31 00:00:00,NaN,1.374000,1.000000,5.0
25%,2013-02-28 00:00:00,NaN,24.990000,1.000000,5.0
50%,2013-09-30 00:00:00,NaN,134.982000,1.000000,5.0
75%,2014-01-31 00:00:00,NaN,1120.490000,3.000000,5.0
max,2014-06-30 00:00:00,NaN,27893.619000,44.000000,5.0


## Summary

This notebook defines `clean_sales_dataframe` and demonstrates its use on
AdventureWorks sales-line data. The function:

- Enforces a datetime `OrderDate`.
- Coerces `LineTotal` and `OrderQty` to numeric.
- Drops rows with invalid dates or fully-missing numeric values.
- Optionally excludes cancelled orders (`Status = 6`).


In [8]:
assert clean_df_sales_lines["OrderDate"].notna().all()
assert clean_df_sales_lines[["LineTotal", "OrderQty"]].notna().any(axis=1).all()

In [11]:
# save cleaned data for reuse in EDA/Power BI

from pathlib import Path

output_path = Path("../data/clean/clean_sales_lines.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

In [12]:
clean_df_sales_lines.to_parquet(output_path, index=False)
output_path

WindowsPath('../data/clean/clean_sales_lines.parquet')